# data_pipeline: Data pipelines for Pre-Training

## Learning Objectives
After studying and implementing this section, you should be able to:

1. Build a streaming data pipeline capable of tokenizing, chunking, shuffling, and batching terabytes of text without loading the entire dataset into memory.
2. Implement data quality filters such as deduplication, language detection, and content filtering, mirroring real-world pre-training pipelines.
3. Construct fixed-length training sequences, properly handling attention masks and document boundaries.
4. Measure and profile pipeline throughput to ensure the DataLoader feeds data fast enough to keep GPU compute fully utilized without starvation.


## What is the problem?
You have a tokenizer; now you need data.

This is not about a small dataset or a CSV file. To train a large language model, you need terabytes of text that has been cleaned, deduplicated, quality-filtered, and evaluated. Then, the text must be tokenized, divided into fixed-length sequences, and delivered to the model in random batches at sufficient speed. The pipeline's speed must be high enough that your eight-GPU cluster never has to wait for the next batch.

Many believe LLM training primarily depends on model architecture, but data plays a decisive role. Llama 3 was trained on 15.6 trillion tokens, GPT-3 on 300 billion tokens, and DeepSeek-V2 on 8.1 trillion tokens. The overall architecture of all three is more or less similar: multiple Transformer blocks stacked together, comprising attention and feed-forward layers. A significant portion of the difference in output quality among these models stems from the volume, composition, and quality of the training data.

DeepMind's Chinchilla paper elaborated on this. For any given compute budget, there is an optimal ratio between the number of model parameters and the number of training tokens. The research indicated that most models in 2022 were significantly undertrained, meaning they had too many parameters relative to the amount of data they were exposed to. For instance, a 70 billion parameter model trained on 1.4 trillion tokens, following the optimal Chinchilla ratio, outperformed Gopher, a 280 billion parameter model trained on only 300 billion tokens.

Therefore, your data pipeline determines whether the model truly learns language, knowledge, and useful patterns, or merely reproduces the noise present in the data.



In [1]:
import re
import hashlib
import random
import time
from collections import Counter, defaultdict
from typing import List, Tuple, Set, Dict, Any, Generator, Optional

In [2]:
# basic clean text
def clean_text(text: str) -> str:
    """
    Cleans raw document text by stripping HTML tags, URLs, non-ASCII characters,
    and normalizing whitespace.

    Args:
        text (str): Input raw text document.

    Returns:
        str: Cleaned and normalized text string.
    """
    # TODO: Strip unwanted markup, non-ASCII noise, and normalize spaces/newlines.

    # if is NOT text-> return ""
    if not text:
        return ""

    # by regex remove tags
    text = re.sub(r"<[^>]+>", " ", text)

    # remove URL
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # remove ASCII
    text = re.sub(r"[^\x00-\x7F]+", " ", text)

    # remove extera space in first and end
    text = re.sub(r"\s+", " ", text).strip()

    return text



Implementing this function in the pre-training pipeline simulates data quality filtering, screening out low-quality inputs such as promotional pages or spam content.

This function must evaluate three criteria:
- Word count
- Ratio of all-uppercase words
- Special character density

This function originates from the logic of well-known pipelines like RefinedWeb and Falcon, which check thresholds for length, uppercase ratio, and punctuation ratio for each document. These three criteria are aggregated to ensure each document passes through three gates.



In [3]:
def quality_filter(
    text: str, 
    min_words: int = 50, 
    max_ratio_caps: int = 0.3, 
    max_ratio_special: float = 0.1
) -> bool:
    """
    Filters out low-quality documents based on length, capitalization ratio, and special character density.

    Args:
        text (str): Cleaned document text.
        min_words (int): Minimum required word count.
        max_ratio_caps (float): Maximum allowed ratio of ALL-CAPS words.
        max_ratio_special (float): Maximum allowed ratio of non-alphanumeric special characters.

    Returns:
        bool: True if the document meets quality criteria, False otherwise.
    """
    # TODO: Check word count thresholds and measure capitalization and special-character ratios.
    
    # min vocab - Word count
    words = text.split()
    if len(words) < min_words:
        return False

    # Ratio of all-uppercase words
    caps_words = sum(
        1 for w in words
        if w.isalpha() and w == w.upper()
    )
    if caps_words / len(words) > max_ratio_caps:
        return False

    # Special character density
    n_special = sum(1 for ch in text if not ch.isalnum())
    if n_special / len(text) > max_ratio_special:
        return False

    return True


This function serves as the foundation for the Deduplication algorithm (removing exact and near-duplicate documents) using MinHash LSH.

### Why Word-based Shingling (Word n-grams)?
- **What is a Shingle?** A continuous sequence of $k$ consecutive words (equivalent to a word-level $n$-gram with length $n = k$).
- **Why a Set?** In MinHash theory, a document is modeled as a "set" of shingles so that the Jaccard Similarity coefficient between two documents $A$ and $B$ can be computed:
  $$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$
- **Why Lowercase?** Variations in letter casing (such as "The" and "the") should not cause identical phrases to be treated as distinct; normalizing to `lower()` is essential.
- **Handling Short Texts:** If the document contains fewer than $k$ words, no shingle of length $k$ can be constructed; in this case, the function must return an empty set (`set()`).


### Implementation Steps
1. Convert the entire text to lowercase using `.lower()`.
2. Tokenize the text into a list of words using `.split()`.
3. Check the length condition: if `len(words) < k`, immediately return `set()`.
4. Iterate over the word list using a sliding window of length $k$, joining the tokens with a single space (`" ".join(...)`) to construct each shingle.
5. Collect and store all generated shingles in a unique `set`.


In [4]:
def get_shingles(text: str, k: int = 5) -> Set[str]:
    """
    Extracts k-shingles (word n-grams) from a text string.

    Args:
        text (str): Input document text.
        k (int): Size of word shingle (n-gram length).

    Returns:
        Set[str]: Unique set of k-word shingles.
    """
    # TODO: Lowercase, tokenize into words, and construct word n-gram shingles.
    if not text:
        return set()

    # Lower
    words = text.lower().split()

    # if k is graten than len(tex) -> cannot use
    if len(words) < k:
        return set()

    # Join
    shingles = {
        " ".join(words[i : i + k]) 
        for i in range(len(words) - k + 1)
    }

    return shingles


This function computes the MinHash signature (a fixed-length vector of size `num_hashes`) for a given set of shingles.

### MinHash Theory:
According to the MinHash theorem, the probability that the minimum hash value of two sets is equal under a random hash function $h_i$ is exactly equal to their Jaccard Similarity:

$$P(h_i(A) = h_i(B)) = J(A, B)$$

Therefore, given $m$ independent hash functions ($h_0, h_1, \dots, h_{m-1}$), the signature vector is constructed as follows:

$$\text{sig}[i] = \min_{s \in \text{shingles}} h_i(s)$$





In [5]:

# shingles from def get_shingles


def minhash_signature(shingles: Set[str], num_hashes: int = 128) -> List[int]:
    """
    Computes MinHash signature for a set of shingles using hash function permutations.

    Args:
        shingles (Set[str]): Set of unique word shingles.
        num_hashes (int): Length of the MinHash signature vector.

    Returns:
        List[int]: MinHash signature list of length (num_hashes,).
    """
    # TODO: Generate hash permutations for each seed and compute the minimum hash value per seed.

    if not shingles:
        return [0] * num_hashes

    signature = []

    for seed in range(num_hashes):
        min_val = float("inf")
        seed_bytes = str(seed).encode("utf-8")

        for s in shingles:
            
            # Combining a seed and a shingle to simulate an independent and deterministic hash function.
            h = hashlib.md5(seed_bytes + b"_" + s.encode("utf-8")).hexdigest()

            #Converting to an integer and optimizing processing speed
            
            val = int(h[:16], 16)

            if val < min_val:
                min_val = val

        signature.append(min_val)

    return signature


If we have $N$ documents, pairwise comparison of MinHash signatures requires $\frac{N(N-1)}{2}$ comparisons, resulting in a time complexity of $O(N^2)$.
The objective of LSH (Locality-Sensitive Hashing) is to reduce these comparisons: it maps documents with a high probability of similarity into a shared bucket, ensuring that only documents within the same bucket are compared.
A document's full signature consists of $M$ hashes (e.g., 128).

- If we enforce that "two documents must match across all 128 hashes," the condition becomes overly strict (retrieving only 100% identical copies).
- If we evaluate "each hash individually," the condition becomes overly relaxed, leading to a high number of dissimilar candidates (False Positives).

**LSH Mathematical Solution:** The signature is divided into $b$ bands, each containing $r$ numbers ($b \times r = M$).

The probability that two documents with Jaccard similarity $s$ become candidate pairs in at least one band is given by:

$$P(\text{Candidate}) = 1 - (1 - s^r)^b$$

This formula produces an S-curve that acts as a sharp threshold filter, identifying documents above the similarity threshold with high probability.

### Why convert to string and then hash with MD5?
```python
        chunk_bytes = ",".join(map(str, chunk)).encode("utf-8")
        bucket_hash = hashlib.md5(chunk_bytes).hexdigest()
```

1. **Exact Matching Within a Band:** We want two documents to be assigned to the same bucket in a band if and only if all $r$ numbers in that band are strictly identical and in the exact same order.
2. **Fixed Size & Dictionary Key:** The sub-vector `chunk` is a Python list of integers. To look up this sub-vector in structures like `dict` or hash tables with $O(1)$ speed while minimizing memory overhead, we map it to a fixed-length hash string (32 Hex characters).


In [6]:
# signature from def minhash_signature

def lsh_buckets(signature: List[int], bands: int = 16) -> List[Tuple[int, str]]:
    """
    Divides MinHash signature into bands and maps each band to a bucket identifier using LSH.

    Args:
        signature (List[int]): MinHash signature vector of shape (num_hashes,).
        bands (int): Number of bands to divide the signature into.

    Returns:
        List[Tuple[int, str]]: List of (band_id, bucket_hash) tuples of length (bands,).
    """
    # TODO: Slice signature into bands, hash each band chunk, and map to bucket identifiers.
    num_hashes = len(signature)
    r = num_hashes // bands

    buckets = []
    for band_id in range(bands):

        chunk = signature[band_id * r : (band_id + 1) * r]

        chunk_bytes = ",".join(map(str, chunk)).encode("utf-8")
        bucket_hash = hashlib.md5(chunk_bytes).hexdigest()

        buckets.append((band_id, bucket_hash))

    return buckets

**Storing `doc_shingles`:**   
We retain the shingles so that in the final step, rather than relying on an estimation, we compute the exact Jaccard similarity. LSH only identifies potential candidate pairs (Candidate Generation); however, because false positives can occur due to random collisions, final filtering must be performed using the exact formula:

$$\frac{|A \cap B|}{|A \cup B|}$$

**Using `defaultdict(list)`:**   
This provides the most efficient approach for grouping documents into LSH buckets. Each key serves as a "bucket address", and its corresponding value is a list of document indices mapped to that bucket.

**Preventing Redundant Computation (`already_compared`):**   
A pair of documents may land in the same bucket across multiple bands. By utilizing a `set` of sorted tuples, we guarantee that each document pair is evaluated for Jaccard similarity only once, preventing unnecessary CPU overhead.

**Removal Strategy (`to_remove`):**   
Instead of removing items dynamically on the fly (which disrupts index alignment), we collect redundant indices into a `set` and reconstruct the entire dataset in a single pass at the end using a list comprehension.

**Note:** In the line `jaccard = len(s1 & s2) / len(s1 | s2)`, a division-by-zero error occurs if both documents are empty. This is prevented using `if not s1 or not s2`.


In [7]:
def deduplicate(
    documents: List[str], 
    threshold: float = 0.8, 
    num_hashes: int = 128, 
    bands: int = 16
) -> Tuple[List[str], int]:
    """
    Finds and removes near-duplicate documents using MinHash LSH and Jaccard similarity evaluation.

    Args:
        documents (List[str]): List of clean document strings.
        threshold (float): Jaccard similarity threshold for considering two docs as duplicates.
        num_hashes (int): Total hash functions for MinHash computation.
        bands (int): Number of bands for LSH partitioning.

    Returns:
        Tuple[List[str], int]: Tuple containing (list of deduplicated documents, count of removed duplicates).
    """
    # TODO: Build MinHash signatures and map documents into LSH band buckets.
    # TODO: Collect candidate pairs from shared buckets, verify Jaccard similarity, and prune duplicates.
    if not documents:
        return [], 0

    # save new data
    # We keep the shingles so that in the final step
    # instead of an estimation, we compute the exact Jaccard similarity
    signatures = []
    doc_shingles = []

    for doc in documents:
        
        shingles = get_shingles(doc, k=5) # from def get_shingles(difalt k=5)
        doc_shingles.append(shingles)
        signatures.append(minhash_signature(shingles, num_hashes))

    # Mapping and make lsh_index
    lsh_index = defaultdict(list)
    for idx, sig in enumerate(signatures):
        buckets = lsh_buckets(sig, bands)
        for band_id, b_hash in buckets:
            lsh_index[(band_id, b_hash)].append(idx)

    # find 
    to_remove = set()
    already_compared = set()

    for candidates in lsh_index.values():
        if len(candidates) < 2:
            continue
            
        # Check all pairs in a bucket
        for i in range(len(candidates)):
            for j in range(i + 1, len(candidates)):
                idx1, idx2 = candidates[i], candidates[j]
                
                # sort for dublicate
                pair = tuple(sorted((idx1, idx2)))
                if pair in already_compared or idx1 in to_remove or idx2 in to_remove:
                    continue
                
                already_compared.add(pair)
                # Verification
                s1, s2 = doc_shingles[idx1], doc_shingles[idx2]
                if not s1 or not s2: continue
                
                jaccard = len(s1 & s2) / len(s1 | s2)
                
                if jaccard >= threshold:
                    to_remove.add(idx2)

    # create list
    deduplicated_docs = [doc for i, doc in enumerate(documents) if i not in to_remove]
    
    return deduplicated_docs, len(to_remove)


The BPE (Byte Pair Encoding) algorithm is a data compression method for language models. The tasks performed in `train_bpe` serve the following purposes:

1. **Compression Efficiency**
Why frequency? We use `stats` to count which character/token pair occurs most frequently in the entire text. By merging the most frequent pair, we shorten the token sequence at each step.
**Goal:** The final model represents similar texts with fewer tokens, increasing speed and reducing memory usage.

2. **Hierarchical Tokenization**
Why perform replacement in the `new_tokens` loop? This is the most critical part. If we only registered merges, we would only learn 2-gram pairs. However, by performing replacement in `tokens`, new tokens are created. In the next step, the algorithm can combine these new tokens with others to form 3-gram, 4-gram, etc. tokens (e.g., a + b -> ab, then ab + c -> abc). This provides a hierarchical structure to the vocabulary.

3. **Managing OOV (Out-Of-Vocabulary)**
Why start from bytes? If we started from Unicode characters, our vocabulary would be excessively large, and we might encounter characters not seen during training. By starting from bytes (0-255), the initial vocabulary space is small and constant, and any string can be represented by these 256 bytes. Thus, OOV errors are effectively eliminated.

4. **Preventing Vocabulary Bloat**
Why the `if stats[best_pair] < 2` condition? If a pair appears only once in the text, merging it does not aid compression. Doing so only consumes vocabulary space without the created token likely being reused. This condition serves as a logical threshold to halt useless merges.


In [ ]:
class SimpleTokenizer:
    """
    A minimal Byte Pair Encoding (BPE) tokenizer implementation operating over byte sequences.
    """
    def __init__(self, vocab_size: int = 256):
        """
        Initializes tokenizer vocabulary and internal mapping structures.

        Args:
            vocab_size (int): Target vocabulary capacity.
        """
        self.vocab: Dict[int, bytes] = {i: bytes([i]) for i in range(256)}
        self.merges: Dict[Tuple[int, int], int] = {}
        self.next_id: int = 256
        self.eos_id: Optional[int] = None
        self.pad_id: int = 0

    def train_bpe(self, text: str, num_merges: int) -> None:
        """
        Iteratively finds the most frequent pair of tokens and merges them into a new token.

        Args:
            text (str): Training text corpus.
            num_merges (int): Number of BPE merge operations to execute.

        Returns:
            None
        """
        # TODO: Convert text to raw byte tokens and iteratively identify most frequent token pairs.
        # TODO: Register new merged tokens into vocabulary and replace occurrences in token stream.
        # TODO: Assign specialized End-of-Sequence (EOS) token ID.
        
        if not text:
            return

        # encode UTF-8
        tokens = list(text.encode("utf-8"))

        for _ in range(num_merges):
            if len(tokens) < 2:
                break

            # zip: fast to run
            # statitics
            stats: Dict[Tuple[int, int], int] = {}
            # add token 1 + token 2 nad zip to make a pair
            for pair in zip(tokens, tokens[1:]):
                stats[pair] = stats.get(pair, 0) + 1

            # if pair have 2 object
            if not stats:
                break

            # max repeite pari
            best_pair = max(stats, key=stats.get)

            # if have NOT 2 object
            if stats[best_pair] < 2:
                break

            # update merges and vocab and id
            idx = self.next_id
            self.merges[best_pair] = idx
            self.vocab[idx] = self.vocab[best_pair[0]] + self.vocab[best_pair[1]]
            self.next_id += 1

            # make new token
            new_tokens = []
            i = 0
            while i < len(tokens):
                if (
                    i < len(tokens) - 1
                    and tokens[i] == best_pair[0]
                    and tokens[i + 1] == best_pair[1]
                ):
                    new_tokens.append(idx)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        self.eos_id = self.next_id
        self.vocab[self.eos_id] = b"<|endoftext|>"
        self.next_id += 1

    def encode(self, text: str) -> List[int]:
        """
        Encodes input text into a list of token IDs using pre-learned BPE merge rules.

        Args:
            text (str): Input text string.

        Returns:
            List[int]: Encoded list of token IDs.
        """
        # TODO: Convert input text to byte IDs and sequentially apply learned BPE merge rules.
        if not text:
            return []

        tokens = list(text.encode("utf-8"))

        # Applying merges in the exact order they were recorded during training
        for pair, new_id in self.merges.items():
            if len(tokens) < 2:
                break

            new_tokens = []
            i = 0
            while i < len(tokens):
                if (
                    i < len(tokens) - 1
                    and tokens[i] == pair[0]
                    and tokens[i + 1] == pair[1]
                ):
                    new_tokens.append(new_id)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        return tokens

    def decode(self, ids: List[int]) -> str:
        """
        Decodes a list of token IDs back into a UTF-8 string.

        Args:
            ids (List[int]): List of token IDs.

        Returns:
            str: Decoded text representation.
        """
        # TODO: Reconstruct byte sequence from token IDs, skipping special tokens, and decode to UTF-8 text.
        # O(n**2) --> O(1)
        byte_sequence = bytearray()
        
        for id_ in ids:
            # Control tokens like `eos_id` are not part of the original text
            # therefore, they must be removed during text reconstruction to avoid introducing noise.
            if self.eos_id is not None and id_ == self.eos_id:
                continue
            
            # Retrieving the bytes corresponding to the ID from the vocabulary
            if id_ in self.vocab:
                byte_sequence.extend(self.vocab[id_])
        
        return byte_sequence.decode("utf-8", errors="replace")

    def vocab_size(self) -> int:
        """
        Returns the total vocabulary size including base bytes and merges.

        Returns:
            int: Total vocabulary size.
        """
        # TODO: Return total number of active entries in vocabulary.
        return len(self.vocab)
